In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm #need to install statsmodels
from statsmodels.stats.anova import anova_lm

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [3]:
df_assess = pd.read_csv("data/cleaned/final_assessment.csv")
df_grades = pd.read_csv("data/cleaned/grades.csv")
df_status = pd.read_csv("data/cleaned/status.csv")

## Merge two dataframes(grades and status) by id, create coding and handwritten binary columns for b1 vs b2 test. I got rid of section 4 for now since they will affect both b1(coefficient of coding) and b2(coefficient of handwritten).

In [4]:
df_status = df_status[df_status['completed'] == 1]

sample = df_grades.merge(df_status[["id", "section"]], on="id", how="inner")

sample['coding'] =  ((sample['section'] == 2) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
sample['handwritten'] =  ((sample['section'] == 3) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
#sample_filter = sample[(sample['coding'] == 0) | (sample['handwritten'] == 0)]
sample

,id,coding_score,handwritten_score,final_score,final_score_adj,section,coding,handwritten
0,0,NaN,NaN,0.737705,0.586066,1,0,0
1,1,0.928571,NaN,0.754098,0.610656,2,1,0
2,3,0.857143,1.0,0.836066,0.479508,4,1,1
3,6,NaN,1.0,0.459016,0.200820,3,0,1
4,9,0.904764,NaN,0.672131,0.471311,2,1,0
5,11,0.928571,1.0,0.918033,0.319672,4,1,1
6,12,NaN,NaN,0.098361,0.000000,1,0,0
7,15,1.000000,1.0,0.344262,0.159836,4,1,1
8,20,NaN,NaN,0.459016,0.274590,1,0,0
9,24,NaN,NaN,1.000000,0.754098,1,0,0


## Linear model for final_score = bias(section1) + b1 * coding + b2 * handwritten

In [5]:
X = sample[['coding', 'handwritten']]
X = sm.add_constant(X)

y = sample['final_score']

model1 = sm.OLS(y, X).fit()

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                 -0.113
Method:                 Least Squares   F-statistic:                    0.1863
Date:                Tue, 10 Feb 2026   Prob (F-statistic):              0.832
Time:                        11:13:13   Log-Likelihood:                 2.1061
No. Observations:                  17   AIC:                             1.788
Df Residuals:                      14   BIC:                             4.287
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.5928      0.094      6.332      

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


## model2 is final score = bias(section 1) + b1 * (handwritten + coding), and we performed a anova between two models. P_value = 0.922997 tells us that we fail to reject the statement that b1(coefficient of coding) = b2 (coefficient of handwritten)

In [7]:
X_2 = sample['coding'] + sample['handwritten']
X_2 = sm.add_constant(X_2)

model2 = sm.OLS(y, X_2).fit()
anova_results = anova_lm(model2, model1)

print(anova_results)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      15.0  0.777437      0.0       NaN       NaN       NaN
1      14.0  0.776899      1.0  0.000537  0.009686  0.922997


## Belows are linear model for adjusted final score, we same x features

In [8]:
y2 = sample['final_score_adj']

adj_model1 = sm.OLS(y2, X).fit()

print(adj_model1.summary())


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                 -0.095
Method:                 Least Squares   F-statistic:                    0.3090
Date:                Tue, 10 Feb 2026   Prob (F-statistic):              0.739
Time:                        11:14:14   Log-Likelihood:                 5.8234
No. Observations:                  17   AIC:                            -5.647
Df Residuals:                      14   BIC:                            -3.147
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.4062      0.075      5.400      

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


## Even though the p value dropped significantly from 0.92 to 0.48, we still failed to reject B1 = B2.

In [9]:
adj_model2 = sm.OLS(y2, X_2).fit()
adj_anova_results = anova_lm(adj_model2, adj_model1)

print(adj_anova_results)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      15.0  0.520641      0.0       NaN       NaN       NaN
1      14.0  0.501692      1.0  0.018949  0.528771  0.479108


## Now we will conduct a hypothesis test for B3 = 0.

In [10]:
sample['both'] = sample['coding'] * sample['handwritten']
X_in = sample[['coding', 'handwritten', 'both']]
X_in = sm.add_constant(X_in)

y_in = sample['final_score']

model1_in = sm.OLS(y_in, X_in).fit()

print(model1_in.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.041
Model:                            OLS   Adj. R-squared:                 -0.181
Method:                 Least Squares   F-statistic:                    0.1837
Date:                Tue, 10 Feb 2026   Prob (F-statistic):              0.906
Time:                        11:14:59   Log-Likelihood:                 2.2359
No. Observations:                  17   AIC:                             3.528
Df Residuals:                      13   BIC:                             6.861
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.5705      0.108      5.258      

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


## Now for adjusted final score

In [11]:
adj_y_in = sample['final_score_adj']

model_adj_in = sm.OLS(adj_y_in, X_in).fit()

print(model_adj_in.summary())

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                 -0.141
Method:                 Least Squares   F-statistic:                    0.3390
Date:                Tue, 10 Feb 2026   Prob (F-statistic):              0.798
Time:                        11:15:10   Log-Likelihood:                 6.0964
No. Observations:                  17   AIC:                            -4.193
Df Residuals:                      13   BIC:                           -0.8600
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.3803      0.086      4.399      

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)
